# 01 — ETL: `public` → `staging`

Crea el esquema `staging` (si no existe) y carga copias limpias y tipadas del raw.

Patrón: cada paso = markdown + celda(s) de ejecución + verificación + gráfica.

## 0. Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.db import read_sql

In [2]:
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
PALETTE = sns.color_palette('crest', as_cmap=False)

## 1. Ejecución del DDL

In [ ]:
%%sql
create schema if not exists staging;

In [ ]:
%%sql
drop table if exists staging.brand cascade;
create table staging.brand as
select  brand_id,
        nullif(trim(name), '')      as name,
        nullif(trim(country), '')   as country,
        nullif(trim(website), '')   as website
from public.brand;

drop table if exists staging.category cascade;
create table staging.category as
select  category_id,
        nullif(trim(name), '')        as name,
        nullif(trim(description), '') as description
from public.category;

drop table if exists staging.product cascade;
create table staging.product as
select  product_id,
        nullif(trim(name), '')         as name,
        nullif(trim(category), '')     as category,
        nullif(trim(manufacturer), '') as manufacturer,
        price,
        created_at::date               as created_date
from public.product;

drop table if exists staging.central_product cascade;
create table staging.central_product as
select  product_id,
        nullif(trim(name), '')   as name,
        category_id,
        brand_id,
        nullif(trim(sku), '')    as sku,
        nullif(trim(barcode), '') as barcode,
        unit_cost,
        unit_price
from public.central_product;

drop table if exists staging.customer cascade;
create table staging.customer as
select  customer_id,
        nullif(trim(first_name), '')                       as first_name,
        nullif(trim(last_name), '')                        as last_name,
        nullif(trim(last_name2), '')                       as last_name2,
        lower(nullif(trim(email), ''))                     as email,
        nullif(regexp_replace(coalesce(phone,''), '\s', '', 'g'), '') as phone,
        created_at::date                                   as signup_date
from public.customer;

drop table if exists staging.city_zone cascade;
create table staging.city_zone as
select  postal_code,
        nullif(trim(district), '')         as district,
        nullif(trim(area_type), '')        as area_type,
        nullif(trim(zone_orientation), '') as zone_orientation,
        nullif(trim(city_code), '')        as city_code,
        nullif(trim(city), '')             as city
from public.city_zone;

drop table if exists staging.store cascade;
create table staging.store as
select  store_id,
        nullif(trim(name), '')        as name,
        nullif(trim(address), '')     as address,
        nullif(trim(city), '')        as city,
        nullif(trim(postal_code), '') as postal_code,
        latitude, longitude,
        opened_date
from public.store;

In [ ]:
%%sql
drop table if exists staging.warehouse cascade;
create table staging.warehouse as
select * from public.warehouse;

drop table if exists staging.warehouse_location cascade;
create table staging.warehouse_location as
select * from public.warehouse_location;

drop table if exists staging.inventory cascade;
create table staging.inventory as
select * from public.inventory;

drop table if exists staging.central_inventory cascade;
create table staging.central_inventory as
select * from public.central_inventory;

drop table if exists staging.offer cascade;
create table staging.offer as
select  offer_id,
        nullif(trim(name), '')        as name,
        nullif(trim(description), '') as description,
        discount_percent,
        start_date,
        end_date
from public.offer;

drop table if exists staging.product_offer cascade;
create table staging.product_offer as
select * from public.product_offer;

drop table if exists staging.return_reason cascade;
create table staging.return_reason as
select  reason_id,
        nullif(trim(reason), '') as reason,
        coalesce(active, true)   as active
from public.return_reason;

drop table if exists staging.return_item cascade;
create table staging.return_item as
select  return_id,
        sale_item_id,
        return_date,
        greatest(quantity, 0) as quantity,
        reason_id
from public.return_item;

drop table if exists staging.sale cascade;
create table staging.sale as
select  sale_id,
        customer_id,
        store_id,
        sale_date,
        coalesce(total, 0) as total
from public.sale
where sale_date is not null;

drop table if exists staging.sale_item cascade;
create table staging.sale_item as
select  sale_item_id,
        sale_id,
        product_id,
        greatest(quantity, 0)         as quantity,
        unit_price,
        offer_id,
        case when subtotal = quantity * unit_price
             then subtotal
             else (greatest(quantity, 0) * unit_price)
        end as subtotal
from public.sale_item;

In [ ]:
%%sql
create index if not exists ix_stg_sale_customer on staging.sale(customer_id);
create index if not exists ix_stg_sale_store    on staging.sale(store_id);
create index if not exists ix_stg_sale_date     on staging.sale(sale_date);
create index if not exists ix_stg_si_sale       on staging.sale_item(sale_id);
create index if not exists ix_stg_si_product    on staging.sale_item(product_id);
create index if not exists ix_stg_ri_sale_item  on staging.return_item(sale_item_id);

## 2. Verificación de carga

### 2.1. Listado de tablas en `staging` — SQL

In [3]:
%%sql
select table_name
from information_schema.tables
where table_schema = 'staging'
order by table_name

,table_name
0,brand
1,category
2,central_inventory
3,central_product
4,city_zone
5,customer
6,inventory
7,offer
8,product
9,product_offer


### 2.2. Conteo exacto por tabla — Python

In [4]:
staging_counts = pd.DataFrame(
    [(t, read_sql(f'select count(*) as n from staging."{t}"')['n'][0])
     for t in staging_tables['table_name']],
    columns=['table', 'rows']
).sort_values('rows', ascending=False).reset_index(drop=True)
staging_counts

,table,rows
0,sale_item,42555
1,sale,20000
2,customer,5750
3,return_item,2330
4,inventory,1000
5,product,50
6,central_inventory,49
7,central_product,49
8,city_zone,42
9,warehouse_location,40


## 3. Comparativa raw vs staging

Confirmamos que las tablas comunes mantienen el mismo número de filas tras la limpieza.

### 3.1. Tablas en `public` — SQL

In [5]:
%%sql
select table_name
from information_schema.tables
where table_schema = 'public'
order by table_name

,table_name
0,brand
1,category
2,central_inventory
3,central_product
4,city_zone
5,customer
6,inventory
7,offer
8,product
9,product_offer


### 3.2. Tabla comparativa raw vs staging — Python

In [6]:
public_counts = pd.DataFrame(
    [(t, read_sql(f'select count(*) as n from public."{t}"')['n'][0])
     for t in public_tables['table_name']],
    columns=['table', 'public_rows']
)
compare = (public_counts
           .merge(staging_counts.rename(columns={'rows':'staging_rows'}),
                  on='table', how='outer')
           .fillna(0).astype({'public_rows':'int64','staging_rows':'int64'}))
compare['delta'] = compare['staging_rows'] - compare['public_rows']
compare = compare.sort_values('public_rows', ascending=False).reset_index(drop=True)
compare

,table,public_rows,staging_rows,delta
0,sale_item,42555,42555,0
1,sale,20000,20000,0
2,customer,5750,5750,0
3,return_item,2330,2330,0
4,inventory,1000,1000,0
5,product,50,50,0
6,central_inventory,49,49,0
7,central_product,49,49,0
8,city_zone,42,42,0
9,warehouse_location,40,40,0
